In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from config.config import RAW_DATA_FILE
from src.data.loader import load_raw_data

df = load_raw_data(RAW_DATA_FILE)
df.shape

(25000, 15)

## Structure

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   delivery_id          25000 non-null  float64
 1   delivery_partner     25000 non-null  str    
 2   package_type         25000 non-null  str    
 3   vehicle_type         25000 non-null  str    
 4   delivery_mode        25000 non-null  str    
 5   region               25000 non-null  str    
 6   weather_condition    25000 non-null  str    
 7   distance_km          25000 non-null  float64
 8   package_weight_kg    25000 non-null  float64
 9   delivery_time_hours  25000 non-null  str    
 10  expected_time_hours  25000 non-null  str    
 11  delayed              25000 non-null  str    
 12  delivery_status      25000 non-null  str    
 13  delivery_rating      25000 non-null  int64  
 14  delivery_cost        25000 non-null  float64
dtypes: float64(4), int64(1), str(10)
memory usage: 

## Missing values

In [3]:
df.isnull().sum()

delivery_id            0
delivery_partner       0
package_type           0
vehicle_type           0
delivery_mode          0
region                 0
weather_condition      0
distance_km            0
package_weight_kg      0
delivery_time_hours    0
expected_time_hours    0
delayed                0
delivery_status        0
delivery_rating        0
delivery_cost          0
dtype: int64

## Full-row duplicates

In [4]:
df.duplicated().sum()

np.int64(0)

## Cardinality of every column

In [5]:
df.nunique().sort_values()

delayed                    2
delivery_status            3
delivery_mode              4
region                     5
delivery_rating            5
vehicle_type               6
weather_condition          6
delivery_partner           9
package_type               9
expected_time_hours        9
delivery_time_hours       20
distance_km             2935
package_weight_kg       4853
delivery_cost          22670
delivery_id            24502
dtype: int64

## Investigate `delivery_id` specifically

In [6]:
print("Unique delivery_id values:", df['delivery_id'].nunique(), "out of", df.shape[0])
dup_id_rows = df[df.duplicated('delivery_id', keep=False)]
print("Rows sharing a delivery_id with at least one other row:", dup_id_rows.shape[0])
dup_id_rows.sort_values('delivery_id').head(6)

Unique delivery_id values: 24502 out of 25000
Rows sharing a delivery_id with at least one other row: 500


,delivery_id,delivery_partner,package_type,vehicle_type,delivery_mode,region,weather_condition,distance_km,package_weight_kg,delivery_time_hours,expected_time_hours,delayed,delivery_status,delivery_rating,delivery_cost
0,250.99,delhivery,automobile parts,bike,same day,west,clear,297.0,46.96,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000008,no,delivered,3,1632.7206
1,250.99,xpressbees,cosmetics,ev van,express,central,cold,89.6,47.39,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003,no,delivered,5,640.1700
2,250.99,shadowfax,groceries,truck,two day,east,rainy,273.5,26.89,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016,no,delivered,4,1448.1700
3,250.99,dhl,electronics,ev van,same day,east,cold,269.7,12.69,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008,no,delivered,3,1486.5700
4,250.99,dhl,clothing,van,two day,north,foggy,256.7,37.02,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000016,no,delivered,4,1394.5600
5,250.99,amazon logistics,documents,ev bike,express,west,rainy,48.4,33.15,1970-01-01 00:00:00.000000004,1970-01-01 00:00:00.000000002,yes,delayed,3,391.4500


## Categorical distributions

In [7]:
categorical_cols = ['delivery_partner', 'package_type', 'vehicle_type',
                     'delivery_mode', 'region', 'weather_condition',
                     'delayed', 'delivery_status']

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


--- delivery_partner ---
delivery_partner
xpressbees          2826
fedex               2818
dhl                 2802
ekart               2801
blue dart           2798
delhivery           2786
shadowfax           2736
ecom express        2722
amazon logistics    2711
Name: count, dtype: int64

--- package_type ---
package_type
fragile items       2848
pharmacy            2810
documents           2805
automobile parts    2795
electronics         2792
clothing            2767
furniture           2746
cosmetics           2744
groceries           2693
Name: count, dtype: int64

--- vehicle_type ---
vehicle_type
ev bike    4218
van        4187
scooter    4174
bike       4160
truck      4145
ev van     4116
Name: count, dtype: int64

--- delivery_mode ---
delivery_mode
two day     6302
same day    6279
express     6233
standard    6186
Name: count, dtype: int64

--- region ---
region
west       5095
central    5060
south      4977
north      4949
east       4919
Name: count, dtype: int64

--

## Numerical summaries

In [8]:
numerical_cols = ['distance_km', 'package_weight_kg', 'delivery_rating', 'delivery_cost']
df[numerical_cols].describe()

,distance_km,package_weight_kg,delivery_rating,delivery_cost
count,25000.000000,25000.000000,25000.000000,25000.000000
mean,150.390436,25.145898,3.666000,864.944579
std,86.409745,14.368663,1.149964,435.712593
min,3.600000,0.670000,1.000000,95.667400
25%,75.900000,12.680000,3.000000,490.800000
50%,151.000000,25.145000,4.000000,867.535000
75%,224.900000,37.660000,5.000000,1237.910000
max,297.100000,49.520000,5.000000,1632.720600


## Inspect the two time columns raw

In [9]:
df[['delivery_time_hours', 'expected_time_hours']].head(10)

,delivery_time_hours,expected_time_hours
0,1970-01-01 00:00:00.000000008,1970-01-01 00:00:00.000000008
1,1970-01-01 00:00:00.000000002,1970-01-01 00:00:00.000000003
2,1970-01-01 00:00:00.000000010,1970-01-01 00:00:00.000000016
3,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008
4,1970-01-01 00:00:00.000000009,1970-01-01 00:00:00.000000016
5,1970-01-01 00:00:00.000000004,1970-01-01 00:00:00.000000002
6,1970-01-01 00:00:00.000000006,1970-01-01 00:00:00.000000008
7,1970-01-01 00:00:00.000000004,1970-01-01 00:00:00.000000008
8,1970-01-01 00:00:00.000000005,1970-01-01 00:00:00.000000008
9,1970-01-01 00:00:00.000000003,1970-01-01 00:00:00.000000008


## Decode what's actually inside them

In [10]:
# Both columns are malformed timestamp strings, e.g. '1970-01-01 00:00:00.000000008'
# The real numeric value appears to be the nanosecond digits after the decimal point.
def extract_encoded_value(x):
    return int(x.split('.')[1])

df['dt_hours_raw'] = df['delivery_time_hours'].apply(extract_encoded_value)
df['et_hours_raw'] = df['expected_time_hours'].apply(extract_encoded_value)

print(df['dt_hours_raw'].describe())
print(df['et_hours_raw'].describe())

count    25000.000000
mean         6.248040
std          3.140935
min          0.000000
25%          4.000000
50%          6.000000
75%          8.000000
max         19.000000
Name: dt_hours_raw, dtype: float64
count    25000.000000
mean        13.107680
std          7.559024
min          2.000000
25%          8.000000
50%          8.000000
75%         16.000000
max         24.000000
Name: et_hours_raw, dtype: float64


## Sanity-check the decoding against a variable we trust (distance)

In [11]:
print("Correlation: decoded delivery time vs distance_km:",
      df['dt_hours_raw'].corr(df['distance_km']))
print("Correlation: decoded delivery time vs delivery_cost:",
      df['dt_hours_raw'].corr(df['delivery_cost']))

Correlation: decoded delivery time vs distance_km: 0.6858833649122651
Correlation: decoded delivery time vs delivery_cost: 0.6789110668010881


## Outlier scan (IQR method, diagnostic only, no removal)

In [14]:
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

for col in numerical_cols:
    print(f"{col}: {iqr_outlier_count(df[col])} potential outliers (IQR method)")

distance_km: 0 potential outliers (IQR method)
package_weight_kg: 0 potential outliers (IQR method)
delivery_rating: 0 potential outliers (IQR method)
delivery_cost: 0 potential outliers (IQR method)


## Relationship check: does `delayed` agree with `delivery_status`?

In [13]:
pd.crosstab(df['delayed'], df['delivery_status'])

delivery_status,delayed,delivered,failed
delayed,,,
no,0,18331,0
yes,5341,0,1328


## Data Understanding — Findings

### Structure
- 25,000 rows, 15 columns, no missing values, no fully-duplicated rows.

### `delivery_id` is NOT a reliable unique key
- Only 24,502 of 25,000 values are unique -- 500 rows share a
  `delivery_id` with at least one other row.
- Root cause: `delivery_id` is stored as a float (e.g. `250.99`) rounded
  to 2 decimals, which causes distinct original IDs to collide onto the
  same displayed value. This is a data quality problem to document, not
  a real duplicate-delivery problem -- the underlying records are clearly
  different deliveries (different partner, region, cost, etc.).
- **Decision for Step 6:** do not use `delivery_id` as a merge/dedup key;
  treat each row as one delivery record regardless of `delivery_id`.

### `delivery_time_hours` and `expected_time_hours` are malformed
- Both are stored as broken timestamp strings
  (`1970-01-01 00:00:00.000000008`) instead of numeric hours.
- The digits after the decimal point behave like the real numeric value:
  `delivery_time_hours` decodes to a 0-19 range, `expected_time_hours`
  decodes to a 2-24 range.
- This decoded value correlates meaningfully with `distance_km` (~0.69)
  and `delivery_cost` (~0.68), confirming it represents real delivery
  duration in hours, not noise.
- **Decision for Step 6:** parse both columns into numeric hour values
  using this decoding logic before any time-based feature engineering.

### Categorical columns -- no invalid values found
- `delivery_partner` (9), `package_type` (9), `vehicle_type` (6),
  `delivery_mode` (4), `region` (5), `weather_condition` (6) all have
  clean, consistent category labels -- no typos, casing issues, or stray
  categories observed.
- `delayed` (yes/no) and `delivery_status` (delivered/delayed/failed)
  are logically consistent with each other (confirmed via crosstab):
  every `delayed == 'yes'` row has `delivery_status` of `delayed` or
  `failed`; every `delayed == 'no'` row is `delivered`.

### Numerical columns
- `distance_km`: 3.6-297.1, no negative or zero-distance values.
- `package_weight_kg`: 0.67-49.52, plausible range.
- `delivery_rating`: clean 1-5 integer scale, no invalid values.
- `delivery_cost`: 95.67-1632.72, no negative costs.
- IQR-based outlier scan found [fill in actual counts from the cell
  output above] -- to be reviewed individually in Step 6, not dropped
  automatically.

### Currency / units
- `delivery_cost` has no stated currency unit in the dataset. This will
  be documented as a limitation in the final report rather than assumed.
